In [1]:
import time 
import os
import collections
import numpy as np
import tensorflow as tf

t0= time.time()
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ===== 1. Caricamento EMNIST da tensorflow_datasets (senza TFF) =====
import tensorflow_datasets as tfds

NUM_CLIENTS = 5
NUM_ROUNDS = 10
EPOCHS = 10
BATCH_SIZE = 20
SHUFFLE_BUFFER = 100
PREFETCH_BUFFER = 10

print("Caricamento EMNIST...")
emnist_data = tfds.load("emnist/byclass", split="train", as_supervised=True)

# ===== 2. Preprocessamento e creazione client dataset =====
def preprocess(images, labels):
    images = tf.reshape(images, [-1, 784])
    images = tf.cast(images, tf.float32) / 255.0
    labels = tf.cast(labels, tf.int32)
    return images, labels

def prepare_client_datasets(dataset, num_clients, samples_per_client=100):
    dataset = dataset.shuffle(10000) 
    dataset = dataset.filter(lambda x, y: tf.less(y, 10))  # ⬅️ filtra etichette 0–9
    dataset = dataset.batch(1)
    all_data = list(tfds.as_numpy(dataset))[:samples_per_client * num_clients]

    clients = []
    for i in range(num_clients):
        client_samples = all_data[i*samples_per_client:(i+1)*samples_per_client]
        x_client = np.array([tf.cast(tf.reshape(x, [-1]), tf.float32) / 255.0 for x, _ in client_samples])
        y_client = np.array([y for _, y in client_samples])
        clients.append((x_client, y_client))

    return clients

print(f"time elapsed: {time.time()-t0}")
clients_data = prepare_client_datasets(emnist_data, NUM_CLIENTS)

Serialize message LOG:name range value <class 'range'>
Serialize message LOG:name print value <built-in function print>
Serialize message LOG:name list value <class 'list'>
INSIDE POSTLOAD KERNEL 1
INSIDE POSTLOAD KERNEL 1
INSIDE POSTLOAD KERNEL 1
INSIDE POSTLOAD KERNEL 1
INSIDE POSTLOAD KERNEL 1
INSIDE POSTLOAD KERNEL 1
INSIDE POSTLOAD KERNEL 1
INSIDE POSTLOAD KERNEL 1
user ns 
{'range': <class 'range'>, 'print': <built-in function print>, 'list': <class 'list'>, '__name__': '__main__', '__builtin__': <module 'builtins' (built-in)>, '__builtins__': <module 'builtins' (built-in)>}
Caricamento EMNIST...
Dl Completed...: 0 url [00:00, ? url/s]
Dl Size...: 0 MiB [00:00, ? MiB/s]

Dl Completed...:   0%|          | 0/1 [00:00<?, ? url/s]
Dl Size...: 0 MiB [00:00, ? MiB/s]

Dl Size...:   0%|          | 0/535 [00:00<?, ? MiB/s]

Dl Size...:   0%|          | 1/535 [00:00<08:15,  1.08 MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]
Dl Size...:   0%|          | 2/535 [00:01<04:41,  1.9

In [2]:
import os
import collections
import numpy as np
import tensorflow as tf

# ===== 3. Definizione modello base =====
def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(784,)),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='sgd',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def get_model_weights(model):
    return model.get_weights()

def set_model_weights(model, weights):
    model.set_weights(weights)

def average_weights(weights_list):
    avg_weights = []
    for weights in zip(*weights_list):
        avg_weights.append(np.mean(weights, axis=0))
    return avg_weights

# ===== 4. Ciclo federato manuale =====
global_model = create_model()

for round_num in range(1, NUM_ROUNDS + 1):
    print(f"\n🔁 Federated Round {round_num}")

    client_weights = []

    for client_idx, (x, y) in enumerate(clients_data):
        client_model = create_model()
        set_model_weights(client_model, get_model_weights(global_model))

        client_model.fit(x, y,
                         batch_size=BATCH_SIZE,
                         epochs=EPOCHS,
                         verbose=0)

        client_weights.append(get_model_weights(client_model))

    new_global_weights = average_weights(client_weights)
    set_model_weights(global_model, new_global_weights)

    # Valutazione su tutti i dati (centrale, per monitoraggio)
    all_x = np.concatenate([x for x, _ in clients_data])
    all_y = np.concatenate([y for _, y in clients_data])
    loss, acc = global_model.evaluate(all_x, all_y, verbose=0)
    print(f"✅ Global Accuracy after round {round_num}: {acc:.4f}")
    print(f"✅ Global Loss after round {round_num}: {loss:.4f}")

Serialize message LOG:name NUM_CLIENTS value 5
Serialize message LOG:name NUM_ROUNDS value 10
Serialize message LOG:name EPOCHS value 10
Serialize message LOG:name BATCH_SIZE value 20
Serialize message LOG:name SHUFFLE_BUFFER value 100
Serialize message LOG:name PREFETCH_BUFFER value 10
Serialize message LOG:name clients_data value [(array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32), array([[6],
       [3],
       [0],
       [0],
       [6],
       [3],
       [6],
       [7],
       [7],
       [5],
       [9],
       [9],
       [7],
       [6],
       [1],
       [2],
       [5],
       [2],
       [7],
       [9],
       [3],
       [2],
       [4],
       [9],
       [6],
       [0],
       [8],
       [3],
       [5],
       [4],
       [2],
       [5],
       [6],
       [5],

In [3]:
import matplotlib.pyplot as plt

# Prendiamo il primo elemento del dataset originale (non preprocessato)
for example_image, example_label in emnist_data.take(1):
    image = example_image.numpy()  # converte da tensore a numpy array
    label = example_label.numpy()

plt.imshow(image, cmap='gray', aspect='equal')
plt.title(f"Label: {label}")
plt.grid(False)
plt.show()


NameError: name 'emnist_data' is not defined